In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, List, Sequence, Dict
import time

from sklearn.model_selection import GroupKFold
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV, calibration_curve # <--- 추가
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
# ⬇️ --- 추가된 모듈 --- ⬇️
from sklearn.metrics import roc_auc_score, brier_score_loss, make_scorer 

# 파이썬의 표준 입/출력 인코딩을 UTF-8로 강제 설정
os.environ["PYTHONIOENCODING"] = "utf-8"

# (선택적) Jupyter/IPython 환경에서 stdout을 강제로 재설정
if 'ipykernel' in sys.modules:
    import io
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
    sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding='utf-8')

print("✅ UTF-8 인코딩 설정 완료")
warnings.filterwarnings("ignore")
pd.set_option('display.max_rows', 100)

# --- 학습 스크립트의 상수 (경로/설정) ---
# (이하 동일)
DATA_DIR = "data"
MODEL_DIR = "model"
N_SPLITS_KFold = 5
RANDOM_STATE = 42

A_MODEL_PATH_TPL = os.path.join(MODEL_DIR, "model_A_fold{fold}.joblib")
B_MODEL_PATH_TPL = os.path.join(MODEL_DIR, "model_B_fold{fold}.joblib")
A_PREPROC_PATH_TPL = os.path.join(MODEL_DIR, "preproc_A_fold{fold}.joblib")
B_PREPROC_PATH_TPL = os.path.join(MODEL_DIR, "preproc_B_fold{fold}.joblib")

# --- joblib.load를 위한 클래스 정의 ---
# (이하 동일)
class AvgProbaEnsemble:
    def __init__(self, models: List):
        self.models = models

    def predict_proba(self, X):
        probs = [m.predict_proba(X) for m in self.models]
        return np.mean(probs, axis=0)

print("라이브러리 및 클래스 정의 완료.")

라이브러리 및 클래스 정의 완료.


In [5]:
# -------------------\
# 데이터 로드 유틸
# -------------------\
def read_index_files() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_idx = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
    test_idx  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
    return train_idx, test_idx

def read_feature_files(split: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    A_df = pd.read_csv(os.path.join(DATA_DIR, split, "A.csv"), low_memory=False)
    B_df = pd.read_csv(os.path.join(DATA_DIR, split, "B.csv"), low_memory=False)
    return A_df, B_df

# -------------------\
# 피처 엔지니어링 유틸
# -------------------\
def safe_split_to_float(series: pd.Series) -> pd.DataFrame:
    """콤마로 구분된 문자열 Series를 파싱하여 float DataFrame으로 반환 (벡터화)"""
    return series.str.split(',', expand=True).astype(float)

def add_rowwise_features(df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
    X = df[feature_cols]
    na_count = X.isna().sum(axis=1).astype(np.int32)
    na_ratio = (na_count / (len(feature_cols) + 1e-9)).astype(np.float32)
    df2 = df.copy()
    df2["NA_COUNT"] = na_count
    df2["NA_RATIO"] = na_ratio
    return df2

# -------------------\
# [V17] 특징 공학 (B검사 '규칙 기반' 피처 추가)
# (제공해주신 원본 스크립트의 create_features 함수 전체)
# -------------------\
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    df_proc = df.copy()

    # 1. Age (나이) 수치화
    age_map = {f"{i}{s}": (i + 2 if s == 'a' else i + 7) for i in range(10, 90, 10) for s in ['a', 'b']}
    age_map.update({
        '10a': 12, '10b': 17, '90a': 92, '90b': 97, '100a': 102
    })
    df_proc['Age_numeric'] = df_proc['Age'].map(age_map).astype(float)

    # 2. TestDate (검사일) 분해
    df_proc['TestDate_num'] = pd.to_numeric(df_proc['TestDate'], errors='coerce')
    df_proc['TestYear'] = (df_proc['TestDate_num'] // 100).astype(float)
    df_proc['TestMonth'] = (df_proc['TestDate_num'] % 100).astype(float)

    # 3. PrimaryKey (운전자) 기반 변수 (정렬이 중요)
    df_proc = df_proc.sort_values(by=['PrimaryKey', 'TestDate_num'])
    
    g_temp = df_proc.groupby('PrimaryKey') # 임시 g (TestCount 등 기본 피처용)
    
    df_proc['TestCount'] = g_temp['Test_id'].transform('count')
    df_proc['TestSequence'] = g_temp.cumcount() + 1
    df_proc['FirstTestAge'] = g_temp['Age_numeric'].transform('min')
    df_proc['FirstTestYear'] = g_temp['TestYear'].transform('min')
    df_proc['TimeSinceFirstTest_yr'] = df_proc['TestYear'] - df_proc['FirstTestYear']
    
    # 4. [V14] 신규 A검사 피처 저장을 위한 dict
    new_A_features = {}
    
    # 5. [V14] A검사 원시 데이터 파싱 (A.csv에만 존재)
    try:
        print("[Global FE V17] Starting Vectorized Parsing for A-Test...")
        
        df_a1_dist = safe_split_to_float(df_proc['A1-4'])
        new_A_features['A1_Dist_Mean'] = df_a1_dist.mean(axis=1)
        new_A_features['A1_Dist_Std'] = df_a1_dist.std(axis=1) # <-- 논문 핵심
        
        df_a2_dist = safe_split_to_float(df_proc['A2-4'])
        new_A_features['A2_Dist_Mean'] = df_a2_dist.mean(axis=1)
        new_A_features['A2_Dist_Std'] = df_a2_dist.std(axis=1)

        df_a3_rt = safe_split_to_float(df_proc['A3-7'])
        df_a3_type = safe_split_to_float(df_proc['A3-5']) # 1:valid-C, 2:valid-IC, 3:invalid-C, 4:invalid-IC
        new_A_features['A3_RT_Valid_Mean'] = df_a3_rt.where(df_a3_type.isin([1, 2])).mean(axis=1) # <-- 논문 핵심
        new_A_features['A3_RT_Invalid_Mean'] = df_a3_rt.where(df_a3_type.isin([3, 4])).mean(axis=1) # <-- 논문 핵심
        new_A_features['A3_RT_Valid_Std'] = df_a3_rt.where(df_a3_type.isin([1, 2])).std(axis=1)
        new_A_features['A3_RT_Invalid_Std'] = df_a3_rt.where(df_a3_type.isin([3, 4])).std(axis=1)
        a3_valid_correct = (df_a3_type == 1).sum(axis=1)
        a3_valid_total = df_a3_type.isin([1, 2]).sum(axis=1)
        a3_invalid_correct = (df_a3_type == 3).sum(axis=1)
        a3_invalid_total = df_a3_type.isin([3, 4]).sum(axis=1)
        new_A_features['A3_Valid_Correct_Rate'] = a3_valid_correct / (a3_valid_total + 1e-6)
        new_A_features['A3_Invalid_Correct_Rate'] = a3_invalid_correct / (a3_invalid_total + 1e-6)

        df_a4_rt = safe_split_to_float(df_proc['A4-5'])
        df_a4_cond = safe_split_to_float(df_proc['A4-1']) # 1:congruent, 2:incongruent
        df_a4_resp = safe_split_to_float(df_proc['A4-3']) # 1:correct, 2:incorrect
        new_A_features['A4_RT_Congruent_Mean'] = df_a4_rt.where((df_a4_cond == 1) & (df_a4_resp == 1)).mean(axis=1) # <-- 논문 핵심
        new_A_features['A4_RT_Incongruent_Mean'] = df_a4_rt.where((df_a4_cond == 2) & (df_a4_resp == 1)).mean(axis=1) # <-- 논문 핵심
        new_A_features['A4_RT_Congruent_Std'] = df_a4_rt.where((df_a4_cond == 1) & (df_a4_resp == 1)).std(axis=1)
        new_A_features['A4_RT_Incongruent_Std'] = df_a4_rt.where((df_a4_cond == 2) & (df_a4_resp == 1)).std(axis=1)
        a4_con_correct = ((df_a4_cond == 1) & (df_a4_resp == 1)).sum(axis=1)
        a4_con_total = (df_a4_cond == 1).sum(axis=1)
        a4_incon_correct = ((df_a4_cond == 2) & (df_a4_resp == 1)).sum(axis=1)
        a4_incon_total = (df_a4_cond == 2).sum(axis=1)
        new_A_features['A4_Congruent_Correct_Rate'] = a4_con_correct / (a4_con_total + 1e-6)
        new_A_features['A4_Incongruent_Correct_Rate'] = a4_incon_correct / (a4_incon_total + 1e-6)

        df_a5_type = safe_split_to_float(df_proc['A5-1']) # 1:non-change, 2:pos, 3:color, 4:shape
        df_a5_resp = safe_split_to_float(df_proc['A5-2']) # 1:correct, 2:incorrect
        a5_invalid_correct = (df_a5_type.isin([2, 3, 4]) & (df_a5_resp == 1)).sum(axis=1)
        a5_invalid_total = df_a5_type.isin([2, 3, 4]).sum(axis=1)
        a5_valid_correct = ((df_a5_type == 1) & (df_a5_resp == 1)).sum(axis=1)
        a5_valid_total = (df_a5_type == 1).sum(axis=1)
        new_A_features['A5_Invalid_Correct_Rate'] = a5_invalid_correct / (a5_invalid_total + 1e-6) # <-- 논문 핵심
        new_A_features['A5_Valid_Correct_Rate'] = a5_valid_correct / (a5_valid_total + 1e-6)

        new_A_features['A6_Correct_Rate'] = pd.to_numeric(df_proc['A6-1'], errors='coerce') / 14.0
        new_A_features['A7_Correct_Rate'] = pd.to_numeric(df_proc['A7-1'], errors='coerce') / 18.0
        
        df_new_A_features = pd.DataFrame(new_A_features, index=df_proc.index)
        df_proc = pd.concat([df_proc, df_new_A_features], axis=1)
        print(f"[Global FE V17] {len(new_A_features)} A-Test paper-based features created.")
    
    except KeyError as e:
        print(f"[Global FE V17] Skipping A-Test parsing (likely B-Test data): {e}")
    except Exception as e:
        print(f"[Global FE V17] ERROR during A-Test parsing: {e}")

    # 6. [V16] 신규 B검사 (선별된) 피처 저장을 위한 dict
    new_B_features = {}

    # 7. [V16] B검사 원시 데이터 파싱 (B.csv에만 존재) - '선별' 버전
    try:
        print("[Global FE V17] Starting Vectorized Parsing for B-Test (Selective)...")
        
        df_b1_rt = safe_split_to_float(df_proc['B1-2'])
        new_B_features['B1_RT_Mean'] = df_b1_rt.mean(axis=1)
        new_B_features['B1_RT_Std'] = df_b1_rt.std(axis=1) # <-- KEEP

        df_b2_rt = safe_split_to_float(df_proc['B2-2'])
        new_B_features['B2_RT_Mean'] = df_b2_rt.mean(axis=1)
        new_B_features['B2_RT_Std'] = df_b2_rt.std(axis=1) # <-- KEEP
        
        df_b3_rt = safe_split_to_float(df_proc['B3-2'])
        new_B_features['B3_RT_Mean'] = df_b3_rt.mean(axis=1)
        new_B_features['B3_RT_Std'] = df_b3_rt.std(axis=1) # <-- KEEP

        # B4 (선택적 주의력) - A4와 동일하게 파싱 (KEEP ALL)
        df_b4_rt = safe_split_to_float(df_proc['B4-2'])
        df_b4_resp = safe_split_to_float(df_proc['B4-1'])
        new_B_features['B4_RT_Congruent_Mean'] = df_b4_rt.where(df_b4_resp.isin([1, 2])).mean(axis=1) # <-- KEEP
        new_B_features['B4_RT_Incongruent_Mean'] = df_b4_rt.where(df_b4_resp.isin([3, 4, 5, 6])).mean(axis=1) # <-- KEEP
        new_B_features['B4_RT_Congruent_Std'] = df_b4_rt.where(df_b4_resp.isin([1, 2])).std(axis=1) # <-- KEEP
        new_B_features['B4_RT_Incongruent_Std'] = df_b4_rt.where(df_b4_resp.isin([3, 4, 5, 6])).std(axis=1) # <-- KEEP
        b4_con_correct = (df_b4_resp == 1).sum(axis=1)
        b4_con_total = df_b4_resp.isin([1, 2]).sum(axis=1)
        b4_incon_correct = (df_b4_resp.isin([3, 5])).sum(axis=1) # 3 and 5 correct
        b4_incon_total = df_b4_resp.isin([3, 4, 5, 6]).sum(axis=1)
        new_B_features['B4_Congruent_Correct_Rate'] = b4_con_correct / (b4_con_total + 1e-6) # <-- KEEP
        new_B_features['B4_Incongruent_Correct_Rate'] = b4_incon_correct / (b4_incon_total + 1e-6) # <-- KEEP

        df_b5_rt = safe_split_to_float(df_proc['B5-2'])
        new_B_features['B5_RT_Mean'] = df_b5_rt.mean(axis=1)
        new_B_features['B5_RT_Std'] = df_b5_rt.std(axis=1) # <-- KEEP
        
        df_new_B_features = pd.DataFrame(new_B_features, index=df_proc.index)
        df_proc = pd.concat([df_proc, df_new_B_features], axis=1)
        print(f"[Global FE V17] {len(new_B_features)} B-Test (Selective) features created.")

    except KeyError as e:
        print(f"[Global FE V17] Skipping B-Test parsing (likely A-Test data): {e}")
    except Exception as e:
        print(f"[Global FE V17] ERROR during B-Test parsing: {e}")

    # 8. [V12] A검사 (인성, A9) 파생 변수
    a9_new_cols = ['A9_Stability_Score', 'A9_Stress_Ratio', 'A9_Reality_Stress']
    safe_cols_A = all(c in df_proc.columns for c in ['A9-1', 'A9-2', 'A9-3', 'A9-5'])
    
    if safe_cols_A:
        df_proc['A9_Stability_Score'] = df_proc['A9-1'] + df_proc['A9-2']
        df_proc['A9_Stress_Ratio'] = df_proc['A9-1'] / (df_proc['A9-5'] + 1e-6)
        df_proc['A9_Reality_Stress'] = df_proc['A9-3'] / (df_proc['A9-5'] + 1e-6)
    else:
        for col in a9_new_cols: df_proc[col] = np.nan
    df_proc[a9_new_cols] = df_proc[a9_new_cols].fillna(0.0)

    # 9. [V12] B검사 (다중과제 B9) 파생 변수 (KEEP ALL)
    b9_new_cols = ['B9_hit_rate', 'B9_fa_rate', 'B9_d_prime_proxy', 'B9_visual_error_rate', 'B9_audio_accuracy']
    safe_cols_B9 = all(c in df_proc.columns for c in ['B9-1', 'B9-2', 'B9-3', 'B9-4', 'B9-5'])
    
    if safe_cols_B9:
        B9_AUDIO_TRIALS = 50.0 
        B9_VISUAL_TRIALS = 32.0
        b9_hit_plus_miss = df_proc['B9-1'] + df_proc['B9-2']
        b9_fa_plus_cr = df_proc['B9-3'] + df_proc['B9-4']
        df_proc['B9_hit_rate'] = df_proc['B9-1'] / (b9_hit_plus_miss + 1e-6)
        df_proc['B9_fa_rate'] = df_proc['B9-3'] / (b9_fa_plus_cr + 1e-6)
        df_proc['B9_d_prime_proxy'] = df_proc['B9_hit_rate'] - df_proc['B9_fa_rate']
        df_proc['B9_visual_error_rate'] = df_proc['B9-5'] / B9_VISUAL_TRIALS
        df_proc['B9_audio_accuracy'] = (df_proc['B9-1'] + df_proc['B9-4']) / B9_AUDIO_TRIALS
    else:
        for col in b9_new_cols: df_proc[col] = np.nan
    df_proc[b9_new_cols] = df_proc[b9_new_cols].fillna(0.0)

    # 10. [V12] B검사 (다중과제 B10) 파생 변수 (KEEP ALL)
    b10_new_cols = ['B10_hit_rate', 'B10_fa_rate', 'B10_d_prime_proxy', 'B10_audio_accuracy', 
                    'B10_vis1_error_rate', 'B10_vis2_accuracy', 'B10_total_visual_error_rate']
    safe_cols_B10 = all(c in df_proc.columns for c in ['B10-1', 'B10-2', 'B10-3', 'B10-4', 'B10-5', 'B10-6'])
    
    if safe_cols_B10:
        B10_AUDIO_TRIALS = 80.0
        B10_VIS1_TRIALS = 52.0
        B10_VIS2_TRIALS = 20.0
        B10_TOTAL_VISUAL_TRIALS = B10_VIS1_TRIALS + B10_VIS2_TRIALS
        b10_hit_plus_miss = df_proc['B10-1'] + df_proc['B10-2']
        b10_fa_plus_cr = df_proc['B10-3'] + df_proc['B10-4']
        df_proc['B10_hit_rate'] = df_proc['B10-1'] / (b10_hit_plus_miss + 1e-6)
        df_proc['B10_fa_rate'] = df_proc['B10-3'] / (b10_fa_plus_cr + 1e-6)
        df_proc['B10_d_prime_proxy'] = df_proc['B10_hit_rate'] - df_proc['B10_fa_rate']
        df_proc['B10_audio_accuracy'] = (df_proc['B10-1'] + df_proc['B10-4']) / B10_AUDIO_TRIALS
        df_proc['B10_vis1_error_rate'] = df_proc['B10-5'] / B10_VIS1_TRIALS
        df_proc['B10_vis2_accuracy'] = df_proc['B10-6'] / B10_VIS2_TRIALS
        b10_total_visual_errors = df_proc['B10-5'] + (B10_VIS2_TRIALS - df_proc['B10-6'])
        df_proc['B10_total_visual_error_rate'] = b10_total_visual_errors / B10_TOTAL_VISUAL_TRIALS
    else:
        for col in b10_new_cols: df_proc[col] = np.nan
    df_proc[b10_new_cols] = df_proc[b10_new_cols].fillna(0.0)

    # 11. [V12] Row-wise NA (결측치) 변수
    base_feature_cols = [c for c in df_proc.columns if (c.startswith("A") or c.startswith("B")) and '-' in c]
    df_proc = add_rowwise_features(df_proc, base_feature_cols)

    # [V12] 'g' 객체를 NA_COUNT 등이 추가된 'df_proc'로 새로고침
    g = df_proc.groupby('PrimaryKey') 

    # 12. [V16] Global/Expanding 피처 대상 컬럼 재정의
    paper_A_features = list(new_A_features.keys())
    paper_B_features = list(new_B_features.keys()) # [V16] 선별된 B 피처 리스트
    
    key_numeric_cols = (
        paper_A_features + # [V14] 신규 A검사 피처
        paper_B_features + # [V16] 신규 (선별된) B검사 피처
        a9_new_cols + 
        b9_new_cols + 
        b10_new_cols + 
        ['NA_COUNT', 'NA_RATIO', 'Age_numeric']
    )
    key_numeric_cols = [c for c in key_numeric_cols if c in df_proc.columns]

    # 13. [V12] Global (전체) 및 Expanding (누적) 통계 피처
    print(f"[Global FE V17] Creating {len(key_numeric_cols)} Global/Expanding features...")
    for col in key_numeric_cols:
        # Global (전체) 통계
        global_mean = g[col].transform('mean')
        global_std = g[col].transform('std')
        
        df_proc[f'{col}_global_mean'] = global_mean
        df_proc[f'{col}_global_std'] = global_std
        
        # 현재 값 vs 전체 평균
        df_proc[f'{col}_vs_global_mean'] = df_proc[col] - global_mean

        # Expanding (누적) 통계
        exp_mean = g[col].expanding(min_periods=1).mean()
        exp_std = g[col].expanding(min_periods=1).std()
        
        df_proc[f'{col}_exp_mean'] = exp_mean.reset_index(level=0, drop=True)
        df_proc[f'{col}_exp_std'] = exp_std.reset_index(level=0, drop=True)

    print("[Global FE V17] Global/Expanding features created.")
    
    # 14. [V17] 규칙 기반 (Z-Score & Flag) 피처 생성
    new_flag_features = []
    try:
        print("[Global FE V17] Creating Rule-Based (Z-Score, Flag, Count) features...")
        key_B_metrics_for_flags = paper_B_features + b9_new_cols + b10_new_cols
        key_B_metrics_for_flags = [c for c in key_B_metrics_for_flags if c in df_proc.columns]

        # "높을수록 나쁨" (반응시간, 편차, 에러율, 오탐지율)
        high_is_bad_cols = [c for c in key_B_metrics_for_flags if 'RT' in c or 'Std' in c or 'error_rate' in c or 'fa_rate' in c]
        # "낮을수록 나쁨" (정답률, 명중률, 정확도, d-prime)
        low_is_bad_cols = [c for c in key_B_metrics_for_flags if 'Correct_Rate' in c or 'hit_rate' in c or 'accuracy' in c or 'd_prime' in c or 'vis2_accuracy' in c]

        bad_flag_sum = pd.Series(0, index=df_proc.index)
        Z_THRESHOLD = 1.5 # 1.5 표준편차 이상 벗어나면 "불량"으로 간주 (튜닝 가능)

        for col in key_B_metrics_for_flags:
            z_score_col = f'{col}_z_score'
            # Z-점수: (현재값 - 개인평균) / 개인표준편차
            df_proc[z_score_col] = (df_proc[col] - df_proc[f'{col}_global_mean']) / (df_proc[f'{col}_global_std'] + 1e-6)
            new_flag_features.append(z_score_col)

            if col in high_is_bad_cols:
                flag_col = f'{col}_is_bad'
                df_proc[flag_col] = (df_proc[z_score_col] > Z_THRESHOLD).astype(int)
                bad_flag_sum += df_proc[flag_col]
                new_flag_features.append(flag_col)
            elif col in low_is_bad_cols:
                flag_col = f'{col}_is_bad'
                df_proc[flag_col] = (df_proc[z_score_col] < -Z_THRESHOLD).astype(int)
                bad_flag_sum += df_proc[flag_col]
                new_flag_features.append(flag_col)

        df_proc['Bad_Test_Count'] = bad_flag_sum # <-- 핵심 피처
        new_flag_features.append('Bad_Test_Count')
        print(f"[Global FE V17] {len(new_flag_features)} rule-based flag/z-score features created.")
    
    except KeyError as e:
        print(f"[Global FE V17] Skipping Rule-Based features (likely A-Test data): {e}")
    except Exception as e:
        print(f"[Global FE V17] ERROR during Rule-Based feature creation: {e}")


    # 15. [V17] 시계열 피처 (Trend) 생성
    # [V17] key_numeric_cols에 Z-Score, Flag 피처들 추가
    key_numeric_cols = key_numeric_cols + new_flag_features
    key_numeric_cols = [c for c in key_numeric_cols if c in df_proc.columns] # 중복 및 오류 방지
    
    print(f"[Global FE V17] Creating {len(key_numeric_cols)} time-series features (diff/shift/roll)...")
    
    for col in key_numeric_cols:
        # diff, shift, roll은 이미 global/exp 피처가 계산된 후이므로 z-score 등에 대해선 계산하지 않음
        # (계산 로직이 꼬일 수 있으므로 v16과 동일하게 유지)
        if col not in new_flag_features: # [V17] 신규 플래그 피처는 시계열 확장 제외
            df_proc[f'{col}_diff'] = g[col].diff()
            df_proc[f'{col}_shift1'] = g[col].shift(1)
            roll_mean = g[col].rolling(3, min_periods=1).mean()
            df_proc[f'{col}_roll3_mean'] = roll_mean.reset_index(level=0, drop=True)
    
    print("[Global FE V17] Time-series features created.")
    
    # ---
    
    df_proc = df_proc.drop(columns=['Age', 'TestDate', 'TestDate_num', 'FirstTestYear'], errors='ignore')
    
    # [V14] 파싱에 사용된 원본 object 컬럼 제거
    raw_a_cols = [
        'A1-1', 'A1-2', 'A1-3', 'A1-4',
        'A2-1', 'A2-2', 'A2-3', 'A2-4',
        'A3-1', 'A3-2', 'A3-3', 'A3-4', 'A3-5', 'A3-6', 'A3-7',
        'A4-1', 'A4-2', 'A4-3', 'A4-4', 'A4-5',
        'A5-1', 'A5-2', 'A5-3'
    ]
    # [V15] B검사 원본 컬럼 제거 목록 업데이트
    raw_b_cols = [
        'B1-1', 'B1-2', 'B1-3',
        'B2-1', 'B2-2', 'B2-3',
        'B3-1', 'B3-2',
        'B4-1', 'B4-2',
        'B5-1', 'B5-2',
        'B6', 'B7', 'B8'
    ]
    df_proc = df_proc.drop(columns=raw_a_cols + raw_b_cols, errors='ignore')
    print(f"[Global FE V17] Dropped {len(raw_a_cols + raw_b_cols)} raw object columns.")
    
    return df_proc

print("헬퍼 함수 정의 완료.")

헬퍼 함수 정의 완료.


In [6]:
print("데이터 로딩 및 전역 피처 생성을 시작합니다...")
t0 = time.time()

label_col = "Label"
key_col = "Test_id"

# 1. 인덱스 및 피처 파일 로드
train_idx, test_idx = read_index_files()
A_train_feat_raw, B_train_feat_raw = read_feature_files("train")
A_test_feat_raw,  B_test_feat_raw  = read_feature_files("test")

# 2. Train/Test 합치기 (Global FE를 위해)
train_feat_raw = pd.concat([A_train_feat_raw, B_train_feat_raw], ignore_index=True)
test_feat_raw  = pd.concat([A_test_feat_raw,  B_test_feat_raw],  ignore_index=True)

all_feat_raw = pd.concat([
    train_feat_raw.assign(is_train=1),
    test_feat_raw.assign(is_train=0)
], ignore_index=True)

# 3. [V17] 전역 피처 생성 함수 호출
all_feat_processed = create_features(all_feat_raw)

# 4. 다시 Train만 분리
train_feat_processed = all_feat_processed[all_feat_processed['is_train'] == 1].drop(columns='is_train')

# 5. A/B 모델별 학습 데이터프레임 준비
A_train_idx = train_idx[train_idx["Test"] == "A"].copy()
B_train_idx = train_idx[train_idx["Test"] == "B"].copy()

# --- A 모델 데이터 ---
df_A_full = A_train_idx.merge(train_feat_processed, on=key_col, how="left", validate="1:1")
y_A = df_A_full[label_col].astype(int).values
groups_A = df_A_full['PrimaryKey'].values
drop_cols_prep = [key_col, label_col, "PrimaryKey", 'Test_x', 'Test_y', "Test"]
X_A_full = df_A_full.drop(columns=drop_cols_prep, errors="ignore")

# --- B 모델 데이터 ---
df_B_full = B_train_idx.merge(train_feat_processed, on=key_col, how="left", validate="1:1")
y_B = df_B_full[label_col].astype(int).values
groups_B = df_B_full['PrimaryKey'].values
X_B_full = df_B_full.drop(columns=drop_cols_prep, errors="ignore")


dt = time.time() - t0
print(f"\n데이터 준비 완료 (소요 시간: {dt:.2f}s)")
print(f"X_A_full shape: {X_A_full.shape} | y_A shape: {y_A.shape}")
print(f"X_B_full shape: {X_B_full.shape} | y_B shape: {y_B.shape}")

데이터 로딩 및 전역 피처 생성을 시작합니다...
[Global FE V17] Starting Vectorized Parsing for A-Test...
[Global FE V17] 20 A-Test paper-based features created.
[Global FE V17] Starting Vectorized Parsing for B-Test (Selective)...
[Global FE V17] 14 B-Test (Selective) features created.
[Global FE V17] Creating 52 Global/Expanding features...
[Global FE V17] Global/Expanding features created.
[Global FE V17] Creating Rule-Based (Z-Score, Flag, Count) features...
[Global FE V17] 53 rule-based flag/z-score features created.
[Global FE V17] Creating 105 time-series features (diff/shift/roll)...
[Global FE V17] Time-series features created.
[Global FE V17] Dropped 38 raw object columns.

데이터 준비 완료 (소요 시간: 2321.03s)
X_A_full shape: (647241, 547) | y_A shape: (647241,)
X_B_full shape: (297526, 547) | y_B shape: (297526,)


In [11]:
# ⬇️ --- [추가] 원본 스크립트의 ECE 계산 함수 --- ⬇️
def expected_calibration_error(y_true, y_prob, n_bins=10):
    # (내용 동일)
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy='uniform')
    bin_totals = np.histogram(y_prob, bins=np.linspace(0, 1, n_bins + 1), density=False)[0]
    non_empty_bins = bin_totals > 0
    bin_weights = bin_totals / len(y_prob)
    bin_weights = bin_weights[non_empty_bins]
    prob_true = prob_true[:len(bin_weights)]
    prob_pred = prob_pred[:len(bin_weights)]
    ece = np.sum(bin_weights * np.abs(prob_true - prob_pred))
    return ece

# ⬇️ --- [수정] 대회 공식 평가지표 함수 --- ⬇️
def competition_score_func(y_true, y_prob, **kwargs): # <--- 여기에 **kwargs 추가
    """대회 공식 평가지표 (낮을수록 좋음)"""
    y_prob_clipped = np.clip(y_prob, 1e-7, 1-1e-7)
    auc = roc_auc_score(y_true, y_prob_clipped)
    brier = brier_score_loss(y_true, y_prob_clipped)
    ece = expected_calibration_error(y_true, y_prob_clipped)
    
    # Optuna에서 사용한 목적 함수 (Minimize)
    combined_score = 0.5 * (1.0 - auc) + 0.25 * brier + 0.25 * ece
    return combined_score

# ⬇️ --- [추가] Sklearn용 커스텀 스코어러 생성 --- ⬇️
# (내용 동일)
competition_scorer = make_scorer(
    competition_score_func, 
    greater_is_better=False, 
    needs_proba=True         
)


def get_fold_importances(which: str, fold_id: int, X_full, y, groups):
    # (이하 함수 내용은 모두 동일합니다)
    """지정된 Fold의 검증 데이터셋에서 Permutation Importance를 계산합니다."""
    
    fold_label = f"[{which} Fold {fold_id+1}/{N_SPLITS_KFold}]"
    print(f"\n--- {fold_label} 중요도 계산 시작 ---")
    
    # 1. 원본과 동일하게 K-Fold 분할
    gkf = GroupKFold(n_splits=N_SPLITS_KFold)
    try:
        train_indices, val_indices = list(gkf.split(X_full, y, groups))[fold_id]
    except IndexError:
        print(f"ERROR: Fold {fold_id}를 분할할 수 없습니다. 스플릿 수({N_SPLITS_KFold})를 확인하세요.")
        return None

    X_val, y_val = X_full.iloc[val_indices], y[val_indices]
    print(f"{fold_label} 검증 데이터 크기: {X_val.shape}")

    # 2. 모델 및 전처리기 로드
    if which == "A":
        MODEL_PATH = A_MODEL_PATH_TPL.format(fold=fold_id)
        PREPROC_PATH = A_PREPROC_PATH_TPL.format(fold=fold_id)
    else:
        MODEL_PATH = B_MODEL_PATH_TPL.format(fold=fold_id)
        PREPROC_PATH = B_PREPROC_PATH_TPL.format(fold=fold_id)

    try:
        preproc = joblib.load(PREPROC_PATH)
        ensemble_model = joblib.load(MODEL_PATH)
    except FileNotFoundError:
        print(f"ERROR: {fold_label} 모델/전처리기 파일을 찾을 수 없습니다. ({MODEL_PATH})")
        return None
        
    # 3. A/B 피처 분리 (학습 때와 동일하게)
    if which == "A":
        cols_to_drop = [c for c in X_val.columns if c.startswith("B")]
        X_val = X_val.drop(columns=cols_to_drop, errors='ignore')
    elif which == "B":
        cols_to_drop = [c for c in X_val.columns if c.startswith("A")]
        X_val = X_val.drop(columns=cols_to_drop, errors='ignore')

    # 4. 데이터 변환
    X_val_t = preproc.transform(X_val)
    feature_names = preproc.get_feature_names_out()
    print(f"{fold_label} 변환된 피처 수: {len(feature_names)}")

    # 5. 앙상블 내 모든 모델(3-seeds)의 중요도 계산
    seed_importances = []
    
    n_repeats = 1 
    
    for i, model_in_ensemble in enumerate(ensemble_model.models):
        print(f"{fold_label} Seed {i+1} 계산 중...")
        
        base_hgb = model_in_ensemble
        if isinstance(base_hgb, CalibratedClassifierCV):
            base_hgb = base_hgb.estimator 

        # Permutation Importance 계산
        perm_imp = permutation_importance(
            base_hgb, 
            X_val_t, 
            y_val, 
            n_repeats=n_repeats, 
            random_state=RANDOM_STATE, 
            scoring=competition_scorer, # <--- 커스텀 스코어러 사용
            n_jobs=-1
        )
        seed_importances.append(perm_imp.importances_mean)

    # 6. 3개 시드의 평균 중요도 반환
    avg_imp = np.mean(seed_importances, axis=0)
    print(f"{fold_label} 계산 완료.")
    
    return pd.Series(avg_imp, index=feature_names)

In [12]:
all_importances_A = []
all_importances_B = []

# --- A 모델 (5-Folds) ---
if len(X_A_full) > 0:
    for fold_id in range(N_SPLITS_KFold):
        imp_A = get_fold_importances("A", fold_id, X_A_full, y_A, groups_A)
        if imp_A is not None:
            all_importances_A.append(imp_A)

# --- B 모델 (5-Folds) ---
if len(X_B_full) > 0:
    for fold_id in range(N_SPLITS_KFold):
        imp_B = get_fold_importances("B", fold_id, X_B_full, y_B, groups_B)
        if imp_B is not None:
            all_importances_B.append(imp_B)

print("\n--- 모든 Fold 중요도 계산 완료 ---")

# --- A 모델 집계 ---
if all_importances_A:
    df_imp_A = pd.concat(all_importances_A, axis=1)
    df_imp_A.columns = [f"Fold_{i+1}" for i in range(len(all_importances_A))]
    df_imp_A["Mean_Imp"] = df_imp_A.mean(axis=1)
    df_imp_A = df_imp_A.sort_values("Mean_Imp", ascending=False)
    
    print("\n--- [모델 A] 상위 50개 피처 중요도 (평균) ---")
    print(df_imp_A.head(50))
    
# --- B 모델 집계 ---
if all_importances_B:
    df_imp_B = pd.concat(all_importances_B, axis=1)
    df_imp_B.columns = [f"Fold_{i+1}" for i in range(len(all_importances_B))]
    df_imp_B["Mean_Imp"] = df_imp_B.mean(axis=1)
    df_imp_B = df_imp_B.sort_values("Mean_Imp", ascending=False)
    
    print("\n--- [모델 B] 상위 50개 피처 중요도 (평균) ---")
    print(df_imp_B.head(50))


--- [A Fold 1/5] 중요도 계산 시작 ---
[A Fold 1/5] 검증 데이터 크기: (129449, 547)
[A Fold 1/5] 변환된 피처 수: 249
[A Fold 1/5] Seed 1 계산 중...


UnicodeEncodeError: 'ascii' codec can't encode characters in position 18-20: ordinal not in range(128)

In [10]:
# --- A 모델 시각화 ---
if 'df_imp_A' in locals() and not df_imp_A.empty:
    plt.figure(figsize=(12, 16))
    top_n = 50
    df_imp_A_top = df_imp_A.iloc[:top_n]
    
    sns.barplot(
        x=df_imp_A_top["Mean_Imp"], 
        y=df_imp_A_top.index
    )
    plt.title(f"[Model A] Top {top_n} Feature Importances (Mean over {len(all_importances_A)} Folds)")
    plt.xlabel("Mean Permutation Importance (AUC drop)")
    plt.ylabel("Features")
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.show()

# --- B 모델 시각화 ---
if 'df_imp_B' in locals() and not df_imp_B.empty:
    plt.figure(figsize=(12, 16))
    top_n = 50
    df_imp_B_top = df_imp_B.iloc[:top_n]
    
    sns.barplot(
        x=df_imp_B_top["Mean_Imp"], 
        y=df_imp_B_top.index
    )
    plt.title(f"[Model B] Top {top_n} Feature Importances (Mean over {len(all_importances_B)} Folds)")
    plt.xlabel("Mean Permutation Importance (AUC drop)")
    plt.ylabel("Features")
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.show()